# 01 — Build recording manifest

This notebook creates an index of the session-1, pre-task EyesClosed and
EyesOpen EEG recordings used in the report.

The output contains one row per recording. This structure supports both:

- paired EO–EC analyses;
- classification of individual recordings.

## Imports and paths

In [1]:
from pathlib import Path
import pandas as pd

# Change only this line if the project folder moves.
PROJECT_ROOT = Path(r"C:\Users\maria\Desktop\EEG-special-course")

RAW_ROOT = PROJECT_ROOT / "data" / "ds005385"
PROCESSED_ROOT = PROJECT_ROOT / "data_processed"
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

PARTICIPANTS_FILE = RAW_ROOT / "participants.tsv"
OUTPUT_FILE = PROCESSED_ROOT / "manifest.csv"

if not RAW_ROOT.exists():
    raise FileNotFoundError(f"Raw-data folder not found: {RAW_ROOT}")

if not PARTICIPANTS_FILE.exists():
    raise FileNotFoundError(
        f"Participant metadata file not found: {PARTICIPANTS_FILE}"
    )

## Load participant metadata

In [2]:
participants = pd.read_csv(PARTICIPANTS_FILE, sep="\t")

metadata_columns = [
    column
    for column in ["participant_id", "age", "sex", "handedness"]
    if column in participants.columns
]

participant_metadata = participants[metadata_columns].copy()

print(f"Participants listed in metadata: {len(participant_metadata)}")
participant_metadata.head()

Participants listed in metadata: 608


,participant_id,age,sex,handedness
0,sub-001,60,F,right
1,sub-002,67,M,right
2,sub-003,44,F,right
3,sub-004,24,F,right
4,sub-005,48,F,right


## Find recordings

In [3]:
rows = []

file_patterns = {
    "EC": "sub-*/ses-1/eeg/*EyesClosed_acq-pre*_eeg.edf",
    "EO": "sub-*/ses-1/eeg/*EyesOpen_acq-pre*_eeg.edf",
}

for condition, pattern in file_patterns.items():
    for eeg_file in sorted(RAW_ROOT.glob(pattern)):
        participant_id = eeg_file.parts[-4]

        relative_eeg_path = eeg_file.relative_to(PROJECT_ROOT)

        rows.append(
            {
                "recording_id": (
                    f"{participant_id}_ses-1_pre_{condition}"
                ),
                "participant_id": participant_id,
                "session": 1,
                "acquisition": "pre",
                "condition": condition,
                "eeg_path": relative_eeg_path.as_posix(),
            }
        )

manifest = pd.DataFrame(rows)

if manifest.empty:
    raise ValueError(
        "No session-1 pre-task EO or EC EDF recordings were found."
    )

manifest = manifest.merge(
    participant_metadata,
    on="participant_id",
    how="left",
    validate="many_to_one",
)

manifest = manifest.sort_values(
    ["participant_id", "condition"]
).reset_index(drop=True)

print(f"Recordings found: {len(manifest)}")
print(f"Participants found: {manifest['participant_id'].nunique()}")

manifest.head()

Recordings found: 1216
Participants found: 608


,recording_id,participant_id,session,acquisition,condition,eeg_path,age,sex,handedness
0,sub-001_ses-1_pre_EC,sub-001,1,pre,EC,data/ds005385/sub-001/ses-1/eeg/sub-001_ses-1_...,60,F,right
1,sub-001_ses-1_pre_EO,sub-001,1,pre,EO,data/ds005385/sub-001/ses-1/eeg/sub-001_ses-1_...,60,F,right
2,sub-002_ses-1_pre_EC,sub-002,1,pre,EC,data/ds005385/sub-002/ses-1/eeg/sub-002_ses-1_...,67,M,right
3,sub-002_ses-1_pre_EO,sub-002,1,pre,EO,data/ds005385/sub-002/ses-1/eeg/sub-002_ses-1_...,67,M,right
4,sub-003_ses-1_pre_EC,sub-003,1,pre,EC,data/ds005385/sub-003/ses-1/eeg/sub-003_ses-1_...,44,F,right


## Add paired-data information

In [4]:
condition_counts = (
    manifest.groupby("participant_id")["condition"]
    .nunique()
    .rename("n_conditions")
)

manifest = manifest.merge(
    condition_counts,
    on="participant_id",
    how="left",
)

manifest["complete_pair"] = manifest["n_conditions"].eq(2)
manifest["pair_id"] = (
    manifest["participant_id"].astype(str)
    + "_ses-1_pre"
)

manifest.head()

,recording_id,participant_id,session,acquisition,condition,eeg_path,age,sex,handedness,n_conditions,complete_pair,pair_id
0,sub-001_ses-1_pre_EC,sub-001,1,pre,EC,data/ds005385/sub-001/ses-1/eeg/sub-001_ses-1_...,60,F,right,2,True,sub-001_ses-1_pre
1,sub-001_ses-1_pre_EO,sub-001,1,pre,EO,data/ds005385/sub-001/ses-1/eeg/sub-001_ses-1_...,60,F,right,2,True,sub-001_ses-1_pre
2,sub-002_ses-1_pre_EC,sub-002,1,pre,EC,data/ds005385/sub-002/ses-1/eeg/sub-002_ses-1_...,67,M,right,2,True,sub-002_ses-1_pre
3,sub-002_ses-1_pre_EO,sub-002,1,pre,EO,data/ds005385/sub-002/ses-1/eeg/sub-002_ses-1_...,67,M,right,2,True,sub-002_ses-1_pre
4,sub-003_ses-1_pre_EC,sub-003,1,pre,EC,data/ds005385/sub-003/ses-1/eeg/sub-003_ses-1_...,44,F,right,2,True,sub-003_ses-1_pre


## Validate the manifest

In [5]:
duplicate_recording_ids = manifest["recording_id"].duplicated().sum()

duplicate_participant_conditions = manifest.duplicated(
    subset=["participant_id", "condition"]
).sum()

missing_paths = sum(
    not (PROJECT_ROOT / Path(path)).exists()
    for path in manifest["eeg_path"]
)

missing_age = (
    manifest["age"].isna().sum()
    if "age" in manifest.columns
    else None
)

missing_sex = (
    manifest["sex"].isna().sum()
    if "sex" in manifest.columns
    else None
)

participants_with_both = (
    manifest.loc[manifest["complete_pair"], "participant_id"]
    .nunique()
)

participants_missing_condition = (
    manifest.loc[~manifest["complete_pair"], "participant_id"]
    .nunique()
)

validation_summary = pd.DataFrame(
    {
        "check": [
            "Number of recordings",
            "Number of participants",
            "Participants with both EO and EC",
            "Participants missing one condition",
            "Duplicate recording IDs",
            "Duplicate participant-condition rows",
            "Missing EEG paths",
            "Missing ages",
            "Missing sex values",
        ],
        "value": [
            len(manifest),
            manifest["participant_id"].nunique(),
            participants_with_both,
            participants_missing_condition,
            duplicate_recording_ids,
            duplicate_participant_conditions,
            missing_paths,
            missing_age,
            missing_sex,
        ],
    }
)

validation_summary

,check,value
0,Number of recordings,1216
1,Number of participants,608
2,Participants with both EO and EC,608
3,Participants missing one condition,0
4,Duplicate recording IDs,0
5,Duplicate participant-condition rows,0
6,Missing EEG paths,0
7,Missing ages,0
8,Missing sex values,0


## Check condition counts

In [6]:
condition_summary = (
    manifest.groupby("condition")
    .agg(
        n_recordings=("recording_id", "size"),
        n_participants=("participant_id", "nunique"),
    )
    .reset_index()
)

condition_summary

,condition,n_recordings,n_participants
0,EC,608,608
1,EO,608,608


## Save manifest

In [7]:
column_order = [
    "recording_id",
    "participant_id",
    "pair_id",
    "session",
    "acquisition",
    "condition",
    "age",
    "sex",
    "handedness",
    "eeg_path",
    "complete_pair",
]

column_order = [
    column for column in column_order
    if column in manifest.columns
]

manifest = manifest[column_order]

manifest.to_csv(OUTPUT_FILE, index=False)

print(f"Saved manifest to: {OUTPUT_FILE}")
print(f"Rows: {len(manifest)}")
print(
    "Complete EO–EC pairs:",
    manifest.loc[manifest["complete_pair"], "participant_id"].nunique(),
)

Saved manifest to: C:\Users\maria\Desktop\EEG-special-course\data_processed\manifest.csv
Rows: 1216
Complete EO–EC pairs: 608
